In [1]:
from spin_lattices import KagomeLattice, SpinLattice
from heisenberg_hamiltonians import HeisenbergJ1J2, SpinSystem
from boolean_analysis import (
    BooleanFourierAnalyzer,
    keep_largest_n,
    keep_everything,
    ScorerType,
    get_scorer,
    SignalOption,
    AmplitudeMedianBinSignalKind,
    SignSignalKind,
    AmplitudeSignalKind,
    SignalKind,
)
from boolean_fourier_learner import BooleanFourierLearner

from pathlib import Path
import numpy as np
import pandas as pd
import lattice_symmetries as ls
import matplotlib.pyplot as plt
from heisenberg_hamiltonians import batched_state_info_df
from itertools import product
import numpy.typing as npt
from tqdm import tqdm
import seaborn as sns
import parse

from parity import popcount, parity

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data
from utils import make_unpacked_configurations

from pytorchtools import EarlyStopping

import pickle
from spin_nn import SpinNN
import datetime

ground_state_cache_dir = Path("groundstates")
fourier_learners_cache_dir = Path("fourier_learners_cache")
experiments_dir = Path("experiments") / "kagome-24-nn-2023-01-27"
experiments_dir.mkdir(parents=True, exist_ok=True)

2023-01-30 15:34:29.590 | DEBUG    | lattice_symmetries:__init__:49 - Initializing Haskell runtime...
2023-01-30 15:34:29.593 | DEBUG    | lattice_symmetries:__init__:51 - Initializing Chapel runtime...
[Debug]   [2023-01-30 15:34:29.643 | DEBUG    | lattice_symmetries:__init__:53 - Setting Python exception handler...
LOCALE0]   Initializing chpl_kernels ...
set_python_exception_handler ...


In [29]:
class FC1SpinNN(SpinNN):
    def __init__(self, lattice: SpinLattice, hidden_size: int):
        super().__init__(lattice)
        input_size = lattice.number_spins
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, 2)

    def forward(self, inp: torch.Tensor) -> torch.Tensor:
        x = self.preprocess(inp)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return  self.postprocess(x)
        

In [30]:
J2 = 0.8
system = HeisenbergJ1J2(
    lattice=KagomeLattice(width=2, height=4),
    J1=1,
    J2=J2,
    use_symmetries=True,
    spin_inversion=1,
    ground_state_cache_dir=ground_state_cache_dir,
    show_progress=True,
)
system.get_eigenstates(1)

number_spins=24
Symmetry group contains 16 elements
Hilbert space dimension is 85662
Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-KagomeLattice2x4-1.0-0.8-True-1-1.pickle
Ground state energy is -40.5183420067


[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...


(array([-40.51834201]),
 array([[ 8.36163038e-08],
        [ 1.13414544e-07],
        [ 1.79331566e-08],
        ...,
        [-8.66655973e-03],
        [-1.17282633e-02],
        [ 3.99580256e-03]]))

In [31]:
df = (
    system.get_df_ground_state(
        canonical_basis=True,
    )
    .assign(
        sign=(lambda df: np.sign(df["eigenstate_coeff"])),
        prob=(lambda df: np.abs(df["eigenstate_coeff"]) ** 2),
    )
    .assign(y=lambda df: (df["sign"] == 1).astype(int))
)

In [32]:
df_rep = df.join(system.lattice.get_state_info_df(hamming_weight=system.lattice.number_spins // 2)).groupby('representative').agg({'sign': 'mean', 'prob': 'sum', 'y': 'mean'})

[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...


In [33]:
eps_train = 1e-2
val_eps = 1e-2
test_eps = 1e-2
batch_size = 64

df_train = df_rep.sample(frac=eps_train, weights="prob")
df_rep_for_val = df_rep.drop(df_train.index)
df_val = df_rep_for_val.sample(frac=val_eps, weights="prob")
df_rep_for_test = df_rep_for_val.drop(df_val.index)
df_test = df_rep_for_test.sample(frac=test_eps, weights="prob")

n_batches = int(np.ceil(len(df_train) / batch_size))
epochs = 20000


In [34]:
df_train

,sign,prob,y
representative,,,
3585610,-1.0,0.001282,0.0
3781457,1.0,0.000315,1.0
325181,-1.0,0.000033,0.0
1701036,1.0,0.000619,1.0
7582022,1.0,0.000044,1.0
...,...,...,...
1225678,-1.0,0.000364,0.0
3589986,-1.0,0.000074,0.0
1787738,1.0,0.000018,1.0


In [35]:
net = FC1SpinNN(lattice=system.lattice, hidden_size=64)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)


In [36]:
def get_inputs_and_labels(df: pd.DataFrame) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    X = torch.tensor(
        make_unpacked_configurations(np.asarray(df.index, dtype="uint64"), number_spins=system.lattice.number_spins).astype('float32')
    )
    y = torch.tensor(df["y"].values.astype("int8"), dtype=torch.long)
    probs = torch.tensor(df["prob"].values.astype("float"), dtype=torch.float32)
    return X, y, probs

inputs_val, labels_val, probs_val = get_inputs_and_labels(df_val)


In [37]:
def evaluate(net, inputs, labels, probs):
    with torch.no_grad():
        outputs = net(inputs)
        _, predicted = torch.max(outputs.data, 1)
        correct = (predicted == labels).sum().item()
        accuracy = correct / len(labels)

        sign_overlap = (
            ((predicted * 2 - 1) * (labels * 2 - 1) * probs).sum() / probs.sum()
        ).item()
        return accuracy, sign_overlap


In [38]:
out = net(
    torch.Tensor(
        make_unpacked_configurations(
            np.asarray(
                net.lattice.get_state_info_df(hamming_weight=net.lattice.number_spins // 2)
                .query("representative == 23550")
                .index
            ),
            net.lattice.number_spins,
        ).astype("float32")
    )
)
assert torch.isclose(out, out[0]).all()


In [39]:
early_stopping = EarlyStopping(patience=1000, delta=0.01, verbose=True)

In [40]:
for epoch in range(epochs):  # loop over the dataset multiple times

    running_loss = 0.0
    i = None
    loss = None

    net.train()
    for i in range(n_batches):
        data = df_train.iloc[i * batch_size : (i + 1) * batch_size]
        inputs, labels, probs = get_inputs_and_labels(data)

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()

        optimizer.step()
    net.eval()
    print(f"[{epoch + 1}, {i}] loss: {loss}")

    accuracy, sign_overlap = evaluate(net, inputs, labels, probs)
    print(f"Test set: accuracy: {100 * accuracy} %, sign overlap: {sign_overlap}")

    accuracy_val, sign_overlap_val = evaluate(net, inputs_val, labels_val, probs_val)
    print(f"Validation set: accuracy: {100 * accuracy_val} %, sign overlap: {sign_overlap_val}")
    
    early_stopping(-sign_overlap_val, net)
    if early_stopping.early_stop:
        print("Early stopping")
        break
    
net.load_state_dict(torch.load('checkpoint.pt'))
    
    


[1, 26] loss: 0.6995253562927246
Test set: accuracy: 38.88888888888889 %, sign overlap: -0.27806809544563293
Validation set: accuracy: 49.494949494949495 %, sign overlap: -0.009638186544179916
Validation loss decreased (inf --> 0.009638).  Saving model ...
[2, 26] loss: 0.6944999694824219
Test set: accuracy: 33.33333333333333 %, sign overlap: -0.29237645864486694
Validation set: accuracy: 50.386215092097444 %, sign overlap: -0.05121379718184471
EarlyStopping counter: 1 out of 1000
[3, 26] loss: 0.6948693990707397
Test set: accuracy: 36.11111111111111 %, sign overlap: -0.28326159715652466
Validation set: accuracy: 50.326797385620914 %, sign overlap: -0.04193653538823128
EarlyStopping counter: 2 out of 1000
[4, 26] loss: 0.6943813562393188
Test set: accuracy: 33.33333333333333 %, sign overlap: -0.28696152567863464
Validation set: accuracy: 50.326797385620914 %, sign overlap: -0.03950618952512741
EarlyStopping counter: 3 out of 1000
[5, 26] loss: 0.6940039992332458
Test set: accuracy: 41.

<All keys matched successfully>

In [41]:
evaluate(net, inputs_val, labels_val, probs_val)

(0.7344028520499108, 0.6125897765159607)

In [42]:
evaluate(net, *get_inputs_and_labels(df_test))

(0.6992797118847539, 0.5466772317886353)